In [ ]:
#Imports
%%capture
!pip install pyopenms pandas
import pandas as pd
from pyopenms import MSExperiment, MzMLFile
import os
import glob

In [ ]:
#Group of mzML files for metabolite processing
%%capture
!unzip "/content/part7.zip"

In [ ]:
# Folder containing mzML files
input_folder = "/content/part7"
output_folder = "/content/mzML_csv_outputs"

os.makedirs(output_folder, exist_ok=True)

# Parameters
top_n_peaks = 10       # top N peaks per spectrum
round_mz = 3
round_intensity = 2
min_intensity = 100    # filter very low-intensity peaks

# Get all .mzML files in folder
mzml_files = glob.glob(os.path.join(input_folder, "*.mzML"))
print(f"Found {len(mzml_files)} .mzML files to process.")

# Process each file
for mzml_file in mzml_files:
  print(f"Processing {os.path.basename(mzml_file)}...")

  # Load mzML
  exp = MSExperiment()
  MzMLFile().load(mzml_file, exp)

  data_rows = []

  for spec in exp:
    rt = spec.getRT()
    mz_array, intensity_array = spec.get_peaks()

    # Filter low-intensity peaks
    filtered = [(mz, inten) for mz, inten in zip(mz_array, intensity_array) if inten >= min_intensity]
    mz_array_filtered, intensity_array_filtered = zip(*filtered)

    top_indices = sorted(range(len(intensity_array_filtered)), key=lambda i: intensity_array_filtered[i], reverse=True)[:top_n_peaks]

    for i in top_indices:
      data_rows.append({"retention_time": round(rt, 2),
                        "mz": round(mz_array_filtered[i], round_mz),
                        "intensity": round(intensity_array_filtered[i], round_intensity)})

    csv_filename = os.path.basename(mzml_file).replace(".mzML", "_top10.csv")
    df_out = pd.DataFrame(data_rows)
    df_out.to_csv(os.path.join(output_folder, csv_filename), index=False)
    print(f"Saved {csv_filename} with {len(df_out)} rows.")

print(f"All files processed. CSVs saved in {output_folder}")

Found 11 .mzML files to process.
Processing PHT_20_162_Cort_1900.mzML...
Saved PHT_20_162_Cort_1900_top10.csv with 48337 rows.
Processing PHT_20_157_Cort_Waking.mzML...
Saved PHT_20_157_Cort_Waking_top10.csv with 49337 rows.
Processing PHT_20_162_Cort_1200.mzML...
Saved PHT_20_162_Cort_1200_top10.csv with 48312 rows.
Processing PHT_20_157_Cort_1600.mzML...
Saved PHT_20_157_Cort_1600_top10.csv with 48221 rows.
Processing PHT_20_162_Cort_Waking.mzML...
Saved PHT_20_162_Cort_Waking_top10.csv with 47920 rows.
Processing PHT_20_163_Cort_1000.mzML...
Saved PHT_20_163_Cort_1000_top10.csv with 48535 rows.
Processing PHT_20_163_Cort_Waking.mzML...
Saved PHT_20_163_Cort_Waking_top10.csv with 49224 rows.
Processing PHT_20_162_Cort_1000.mzML...
Saved PHT_20_162_Cort_1000_top10.csv with 48107 rows.
Processing PHT_20_163_Cort_1200.mzML...
Saved PHT_20_163_Cort_1200_top10.csv with 48642 rows.
Processing PHT_20_162_Cort_1600.mzML...
Saved PHT_20_162_Cort_1600_top10.csv with 48920 rows.
Processing PHT_

In [ ]:
#Example mzML file
df = pd.read_csv("/content/mzML_csv_outputs/PHT_20_018_Cort_1000_top10.csv")
df

,retention_time,mz,intensity
0,0.17,102.034,9404713.00
1,0.17,113.964,5839293.00
2,0.17,167.013,5543206.00
3,0.17,158.961,5341787.50
4,0.17,182.985,4836222.00
...,...,...,...
48257,750.12,93.824,1067.97
48258,750.12,71.629,956.17
48259,750.12,54.374,917.19
48260,750.12,50.088,907.92
